# GPW anisotropic heat: with vs without stitches (prior weights)

Local CPU notebook for `experiments/anisotropic_heat.py`. Two joined plates with swapped
diagonal diffusivity; two ARD RBF submodels; four support-cloud orientations.

Agreement weights are the current prior-conditional ones, alpha(x) from
R = Cov(local functions | U, Delta_S=0), independent of q(U). No model change.

Conditions are (stitch_mode, q_rank) pairs:
- `("fixed", 12)`: original setup, 10 fixed stitches at x=0.5, rank-12 V.
- `("none", 0)`: no stitch constraints, no V. Same data, supports and settings otherwise.
- `("independent", 0)`: "split normally": one single-patch GP per plate, each trained only on its
  own plate's observations and predicting only its plate (no fusion, no stitches).

Stitching test design (OVERRIDES): fewer observations, a training-free `gap_width` band on one side
of the interface x=0.5 (`gap_side`), and a longer diffusion `time` so the field crosses the interface.
Metrics are also reported per region: gap band, near interface (|x-0.5| < near_interface_width,
outside the gap) and interior.

Run from the repository's `notebooks/` folder (or repository root) with the project environment.

## 1. Setup

In [ ]:
import json, sys
from dataclasses import asdict, replace
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / "experiments/anisotropic_heat.py").is_file():
    raise RuntimeError(f"Repository root not found from {Path.cwd()}")
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
import pandas as pd
from IPython.display import Image, Markdown, display
from experiments.anisotropic_heat import HeatConfig, run
print("Repository:", ROOT)

## 2. Seeds and conditions

`SEEDS` is editable: replace or add distinct seeds. In this experiment the seed controls
**both** the training/test split and noise **and** model initialization (`HeatConfig.seed`).
Each added seed multiplies compute time by the number of conditions x 4 orientations.

In [ ]:
SEEDS = [31]
CONDITIONS = [("fixed", 12), ("none", 0), ("independent", 0)]  # (stitch_mode, q_rank): "fixed" | "none" | "independent"
OVERRIDES = {"observations": 200, "gap_width": 0.08, "gap_side": "right", "time": 0.04}  # explicit HeatConfig overrides
RUN_TRAINING = False  # set True to train

if not SEEDS or len(set(SEEDS)) != len(SEEDS) or any(type(s) is not int for s in SEEDS):
    raise ValueError("SEEDS must be distinct integers")
if not CONDITIONS or len(set(CONDITIONS)) != len(CONDITIONS):
    raise ValueError("CONDITIONS must be distinct (stitch_mode, q_rank) pairs")
unknown = set(OVERRIDES) - set(asdict(HeatConfig()))
if unknown or {"seed", "stitch_mode", "q_rank"} & set(OVERRIDES):
    raise ValueError(f"Invalid overrides: {sorted(unknown) or 'seed/stitch_mode/q_rank are set by the plan'}")
PLAN = [replace(HeatConfig(), **OVERRIDES, seed=s, stitch_mode=m, q_rank=r)
        for s in SEEDS for m, r in CONDITIONS]
for config in PLAN:  # validate values before training
    if config.stitch_mode not in ("fixed", "none", "independent") or config.q_rank < 0:
        raise ValueError(f"Invalid condition {(config.stitch_mode, config.q_rank)}")
display(pd.DataFrame([{"seed": c.seed, "stitch_mode": c.stitch_mode, "q_rank": c.q_rank,
                       "stitches": c.stitches if c.stitch_mode == "fixed" else 0,
                       "observations": c.observations, "gap_width": c.gap_width, "time": c.time,
                       "epochs": c.epochs, "warmup": c.warmup_epochs,
                       "weight_epsilon": c.weight_epsilon} for c in PLAN]))
print(f"Effective seeds: {SEEDS}; runs: {len(PLAN)} x {len(HeatConfig().orientations)} orientations")

## 3. Train

In [ ]:
if not RUN_TRAINING:
    raise RuntimeError("Training is gated: set RUN_TRAINING=True in the settings cell")
RUN_ROOT = ROOT / "results/anisotropic_heat_stitch_ablation" / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ROOT.mkdir(parents=True, exist_ok=False)
(RUN_ROOT / "plan.json").write_text(json.dumps([asdict(c) for c in PLAN], indent=2))
rows = []
for config in PLAN:
    label = f"seed{config.seed}_{config.stitch_mode}_rank{config.q_rank}"
    display(Markdown(f"### {label}"))
    output = run(config, directory=RUN_ROOT / label)
    for row in json.loads((output / "metrics.json").read_text()):
        rows.append({"condition": f"{config.stitch_mode}/rank{config.q_rank}", "seed": config.seed,
                     "orientation": str(tuple(row["angles_degrees"])), **row})
    pd.DataFrame(rows).to_csv(RUN_ROOT / "comparison.csv", index=False)
    for name in ("points.png", "fits.png"):
        display(Image(filename=str(output / name)))
print("Results:", RUN_ROOT)

## 4. Compare

In [ ]:
table = pd.DataFrame(rows)
for region in ("gap", "near_interface", "interior"):
    for metric in ("rmse_standardized", "nlpd_standardized", "coverage_95"):
        table[f"{region}_{metric}"] = [r["regions"][region][metric] if r["regions"][region] else None for r in rows]
for metric in ("heldout_rmse_standardized", "nlpd_standardized", "coverage_95",
               "gap_rmse_standardized", "gap_nlpd_standardized", "near_interface_rmse_standardized",
               "near_interface_nlpd_standardized", "interior_rmse_standardized"):
    display(Markdown(f"**{metric}** (mean over seeds; sample SD shown when >1 seed)"))
    grouped = table.groupby(["orientation", "condition"])[metric]
    summary = grouped.mean().unstack("condition")
    if len(SEEDS) > 1:
        summary = summary.round(4).astype(str) + " ± " + grouped.std(ddof=1).unstack("condition").round(4).astype(str)
    display(summary)